# Continuous-attractor figures

Figure 3c, its SNR-per-energy inset, and ED Figure 6b.

**Not executed:** simulation run data are not included or available here.
Provide run folders containing `run_metadata.json`, `state_history.npy`, and
`energy_metrics.npy` (or the recorded energy-metrics path). The default run directory
is `../data/continuous_attractor/`; `CALCIUM_RUNS_DIR` can point to an external directory.

Original dynamic analysis/plotting functions and settings are retained. Only input
paths are adapted. Source generation code is provided in `../main_gain.py` and
`../src/`, but the exact paper run configuration must be recovered before regenerating data.
See `../README.md`. No notebook cells, fits, plots, or simulations were run while packaging.


In [ ]:
from pathlib import Path
import sys

_reproduction_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "src" / "reproduction_paths.py").is_file()),
    None,
)
if _reproduction_root is None:
    raise FileNotFoundError("Start Jupyter from the repository root or its notebooks folder.")
sys.path.insert(0, str(_reproduction_root / "src"))
from reproduction_paths import RAW_DATA_DIR, CACHE_DIR, MOVIE_DATA_DIR, CANN_RUNS_DIR, STYLE_PATH


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use(STYLE_PATH)
COLORS = {
    "FR": "#a82203",
    "CTR": "#208cc0",
    "compare": "#f1af3a",
    "Simulation": "#cf5e4e",
    "Theory": "#637b31",
    "Other": "#003967",
    "Purple": "#7B4FA3"
}

def mm_to_inch(mm):
    return mm / 25.4
def clean_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(length=2.5, width=0.6)
    return ax


## Shared state-data loading and SNR calculations

These cells provide the simulation-derived SNR required by all three plots.
The original loader and analysis are copied without executing them.


In [ ]:
from pathlib import Path
from typing import Dict, Tuple, Iterable, Optional, List
import json
import math

RUNS_DIR = Path(CANN_RUNS_DIR)

def build_state_tensors(
    runs_dir: Path = RUNS_DIR,
    ampar_values: Optional[Iterable[float]] = None,
    rin_values: Optional[Iterable[float]] = None,
    eta_values: Optional[Iterable[float]] = None,
    temp_values: Optional[Iterable[float]] = None,
    sigma_theta_values: Optional[Iterable[float]] = None,
    idir_values: Optional[Iterable[float]] = None,
    trials: Optional[Iterable[int]] = None,
    threshold_active_fraction_values: Optional[Iterable[float]] = None,
    active_fraction_values: Optional[Iterable[float]] = None,
    recurrent_width_values: Optional[Iterable[float]] = None,
    I_str_values: Optional[Iterable[float]] = None,
    I_sigma_values: Optional[Iterable[float]] = None,
    strict: bool = True,
    tol: float = 1e-12,
) -> Tuple[
    Dict[Tuple[float, float, float, float, float, float, float], np.ndarray],
    List[float],
    List[float],
    List[float],
    List[float],
    List[float],
    List[float],
    List[int],
    int,
    List[float],
    List[float],
    List[float],
    List[float],
]:
    """Load run metadata and state histories into tensors.

    Returns tensors keyed by
      (ampar, rin, sigma_temp, sigma_eta, sigma_theta, recurrent_width, active_fraction)
    plus parameter axes and network parameter values from metadata.

    Backward compatibility:
      - If new metadata fields are missing, falls back to older fields.
      - threshold_active_fraction_values remains as an alias for active_fraction_values.
    """
    def sorted_unique(values: Iterable[float]) -> List[float]:
        return sorted({v for v in values if v is not None})

    def in_allowed(value, allowed: Optional[Iterable[float]]) -> bool:
        if allowed is None:
            return True
        if value is None:
            return False
        if isinstance(value, (float, int)):
            return any(math.isclose(float(value), float(v), rel_tol=tol, abs_tol=tol) for v in allowed)
        return value in allowed

    def load_state_matrix(folder: Path) -> np.ndarray:
        history_file = folder / "state_history.npy"
        if not history_file.exists():
            raise FileNotFoundError(f"Missing state history file: {history_file}")
        states = np.load(history_file)  # generations x neurons
        if states.ndim != 2:
            raise ValueError(f"Expected 2-D array in {history_file}, got {states.shape}")
        return states.T  # neurons x generations

    def get_active_fraction(spec: dict, net: dict):
        active = spec.get("active_fraction")
        if active is None:
            active = net.get("threshold_active_fraction")
        return active

    def get_recurrent_width(spec: dict, net: dict):
        width = spec.get("cann_width")
        if width is None:
            width = net.get("recurrent_width_fraction")
        if width is None:
            width = spec.get("threshold")
        if width is None:
            width = net.get("threshold_active_fraction")
        return width

    if active_fraction_values is not None and threshold_active_fraction_values is not None:
        lhs = sorted(list(active_fraction_values))
        rhs = sorted(list(threshold_active_fraction_values))
        if len(lhs) != len(rhs) or any(not math.isclose(float(a), float(b), rel_tol=tol, abs_tol=tol) for a, b in zip(lhs, rhs)):
            raise ValueError("Provide only one of active_fraction_values or threshold_active_fraction_values.")
    if active_fraction_values is None and threshold_active_fraction_values is not None:
        active_fraction_values = threshold_active_fraction_values

    meta_paths = sorted(runs_dir.glob("*/run_metadata.json"))
    if not meta_paths:
        raise FileNotFoundError(f"No run_metadata.json found under {runs_dir}")

    metas = [(p.parent, json.loads(p.read_text())) for p in meta_paths]
    specs = [m.get("spec", {}) for _, m in metas]
    net_params = [m.get("network_params", {}) for _, m in metas]

    if ampar_values is None:
        ampar_values = sorted_unique(s.get("ampar") for s in specs)
    else:
        ampar_values = list(ampar_values)
    if rin_values is None:
        rin_values = sorted_unique(s.get("rin") for s in specs)
    else:
        rin_values = list(rin_values)
    if eta_values is None:
        eta_values = sorted_unique(s.get("sigma_eta") for s in specs)
    else:
        eta_values = list(eta_values)
    if temp_values is None:
        temp_values = sorted_unique(s.get("sigma_temp") for s in specs)
    else:
        temp_values = list(temp_values)
    if sigma_theta_values is None:
        sigma_theta_values = sorted_unique(s.get("sigma_theta") for s in specs)
    else:
        sigma_theta_values = list(sigma_theta_values)
    if idir_values is None:
        idir_values = sorted_unique(s.get("idir") for s in specs)
    else:
        idir_values = list(idir_values)
    if trials is None:
        trials = sorted_unique(s.get("trial") for s in specs)
    else:
        trials = list(trials)

    detected_active_fraction_values = sorted_unique(
        get_active_fraction(spec, net) for spec, net in zip(specs, net_params)
    )
    if active_fraction_values is None:
        active_fraction_values = detected_active_fraction_values
        active_fraction_filter = None
    else:
        active_fraction_filter = list(active_fraction_values)
        active_fraction_values = active_fraction_filter

    detected_recurrent_width_values = sorted_unique(
        get_recurrent_width(spec, net) for spec, net in zip(specs, net_params)
    )
    if recurrent_width_values is None:
        recurrent_width_values = detected_recurrent_width_values
        recurrent_width_filter = None
    else:
        recurrent_width_filter = list(recurrent_width_values)
        recurrent_width_values = recurrent_width_filter

    if I_str_values is None:
        I_str_values = sorted_unique(p.get("I_str") for p in net_params)
        I_str_filter = None
    else:
        I_str_filter = list(I_str_values)
        I_str_values = I_str_filter

    if I_sigma_values is None:
        I_sigma_values = sorted_unique(p.get("sigma_input") for p in net_params)
        I_sigma_filter = None
    else:
        I_sigma_filter = list(I_sigma_values)
        I_sigma_values = I_sigma_filter

    _block_sizes = sorted_unique(s.get("block_size") for s in specs)
    BLOCK_SIZE = _block_sizes[0] if _block_sizes else 1
    if len(_block_sizes) > 1:
        print(f"Note: multiple block_size values found: {_block_sizes}")

    runs_by_key: Dict[Tuple[float, float, float, float, float, float, float], Dict[Tuple[float, int], Path]] = {}

    for run_dir, meta in metas:
        spec = meta.get("spec", {})
        net = meta.get("network_params", {})
        ampar = spec.get("ampar")
        rin = spec.get("rin")
        sigma_temp = spec.get("sigma_temp")
        sigma_eta = spec.get("sigma_eta")
        sigma_theta = spec.get("sigma_theta")
        idir = spec.get("idir")
        trial = spec.get("trial")
        active_fraction = get_active_fraction(spec, net)
        recurrent_width = get_recurrent_width(spec, net)
        I_str = net.get("I_str")
        I_sigma = net.get("sigma_input")

        if None in (ampar, rin, sigma_temp, sigma_eta, sigma_theta, idir, trial, active_fraction, recurrent_width):
            continue

        if not in_allowed(ampar, ampar_values):
            continue
        if not in_allowed(rin, rin_values):
            continue
        if not in_allowed(sigma_temp, temp_values):
            continue
        if not in_allowed(sigma_eta, eta_values):
            continue
        if not in_allowed(sigma_theta, sigma_theta_values):
            continue
        if not in_allowed(idir, idir_values):
            continue
        if not in_allowed(trial, trials):
            continue
        if recurrent_width_filter is not None and not in_allowed(recurrent_width, recurrent_width_filter):
            continue
        if active_fraction_filter is not None and not in_allowed(active_fraction, active_fraction_filter):
            continue
        if I_str_filter is not None and not in_allowed(I_str, I_str_filter):
            continue
        if I_sigma_filter is not None and not in_allowed(I_sigma, I_sigma_filter):
            continue

        key = (ampar, rin, sigma_temp, sigma_eta, sigma_theta, recurrent_width, active_fraction)
        runs_by_key.setdefault(key, {})[(idir, trial)] = run_dir

    tensors: Dict[Tuple[float, float, float, float, float, float, float], np.ndarray] = {}
    for key, mapping in runs_by_key.items():
        idir_blocks = []
        for idir in idir_values:
            trial_blocks = []
            for trial in trials:
                folder = mapping.get((idir, trial))
                if folder is None:
                    if strict:
                        raise KeyError(f"Missing run for key={key}, idir={idir}, trial={trial}")
                    continue
                trial_blocks.append(load_state_matrix(folder))
            if not trial_blocks:
                if strict:
                    raise KeyError(f"No trials found for key={key}, idir={idir}")
                continue
            trial_cube = np.stack(trial_blocks, axis=2)  # neurons x generations x trials
            idir_blocks.append(trial_cube)
        if not idir_blocks:
            if strict:
                raise KeyError(f"No idir blocks found for key={key}")
            continue
        tensors[key] = np.stack(idir_blocks, axis=3)  # neurons x generations x trials x idir

    if strict and not tensors:
        raise ValueError(f"No runs matched filters under {runs_dir}")

    return (
        tensors,
        ampar_values,
        rin_values,
        eta_values,
        temp_values,
        sigma_theta_values,
        idir_values,
        trials,
        BLOCK_SIZE,
        recurrent_width_values,
        active_fraction_values,
        I_str_values,
        I_sigma_values,
    )


In [ ]:
(
    tensors,
    AMPAR_VALUES,
    RIN_VALUES,
    ETA_VALUES,
    TEMP_VALUES,
    SIG_THETA_VALUES,
    IDIR_VALUES,
    TRIAL_INDICES,
    BLOCK_SIZE,
    RECURRENT_WIDTH_VALUES,
    ACTIVE_FRACTION_VALUES,
    I_STR_VALUES,
    I_SIGMA_VALUES,
) = build_state_tensors()


In [ ]:
def snr_for_window(state_one_angle, t0, t1):
    X1 = state_one_angle[:, t0:t1, :, 0].mean(axis=1)
    X2 = state_one_angle[:, t0:t1, :, 1].mean(axis=1)

    mu1 = X1.mean(axis=1, keepdims=True)
    mu2 = X2.mean(axis=1, keepdims=True)
    delta = (mu1 - mu2).ravel()

    covariance_reg = np.cov(X1)
    evals, evecs = np.linalg.eigh(covariance_reg)
    evals_reg = np.maximum(evals, 1e-6)
    snr = np.sum((delta*evecs)**2 / evals_reg)

    return snr

TEMP = TEMP_VALUES[-1]
THETA = SIG_THETA_VALUES[-1]
ETA = ETA_VALUES[-1]
window = 5000
stride = 1000  # evaluate every 100 time bins instead of every bin

ctr_snr = []
fr_snr = []
fr_alternate_snr = []
times = []

ctr_state = tensors[1.0, 1.0, TEMP, ETA, THETA,
                    RECURRENT_WIDTH_VALUES[0],
                    ACTIVE_FRACTION_VALUES[0]][:,:,:,:]
fr_alternate = tensors[0.86, 1.00, TEMP, ETA, THETA,
                   RECURRENT_WIDTH_VALUES[0],
                   ACTIVE_FRACTION_VALUES[0]][:,:,:,:]

fr_state = tensors[0.71, 1.22, TEMP, ETA, THETA,
                   RECURRENT_WIDTH_VALUES[0],
                   ACTIVE_FRACTION_VALUES[0]][:,:,:,:]
print(ctr_state.shape)
for t0 in range(0, ctr_state.shape[1] - window + 1, stride):
    t1 = t0 + window
    times.append((t0 + t1) / 2)

    ctr_snr.append(snr_for_window(ctr_state, t0, t1))
    fr_snr.append(snr_for_window(fr_state, t0, t1))
    fr_alternate_snr.append(snr_for_window(fr_alternate, t0, t1))


In [ ]:
times = np.asarray(np.linspace(0,50000, len(ctr_snr)))
ctr_snr = np.asarray(ctr_snr)
fr_snr = np.asarray(fr_snr)
fr_alternate_snr = np.asarray(fr_alternate_snr)
dt = 0.00004  # seconds per time bin
window = 5000
stride = 1000

window_s = window#window#* dt
ctr_snr_rate = ctr_snr / window_s
fr_snr_rate = fr_snr / window_s
fr_alternate_snr_rate = fr_alternate_snr / window_s


## Extended Data Figure 6b: Simulated SNR dynamics and FR alternative

Original CTR, FR, and FR-alternative time-course plot. The source uses
`(AMPAR, RIN) = (1.0, 1.0)`, `(0.71, 1.22)`, and `(0.86, 1.0)`, respectively.


In [ ]:
fig, ax = plt.subplots(
    figsize=(mm_to_inch(45), mm_to_inch(35)),
    dpi=500,
    constrained_layout=True,
)

ax.plot(
    times*dt,
    ctr_snr_rate,
    color=COLORS["CTR"],
    label="CTR",
    lw=1.5,
)

ax.plot(
    times*dt,
    fr_snr_rate,
    color=COLORS["FR"],
    label="FR",
    lw=1.5,
)

ax.plot(
    times*dt,
    fr_alternate_snr_rate,
    color="grey",
    label="FR (alternate)",
    lw=1.5,
)

ax.set_xlabel("Time (s)")
ax.set_ylabel(r"SNR")

ax.legend(frameon=False, handlelength=1.6)
clean_axis(ax)
plt.show()


## Figure 3c: Accuracy versus energy in the continuous-attractor simulation

The following cells load the original energy metrics, fit decision times from the
simulation SNR, and plot accuracy versus normalized energy at the five orientation
separations. The source's normalization, including the final factor of two for
the CTR energy-in-1-s axis, is retained.


In [ ]:
ENERGY_COLUMNS = ("proposal_count", "accepted_count", "sum_abs_total_drive", "mean_abs_total_drive")
ENERGY_COL_IDX = {name: idx for idx, name in enumerate(ENERGY_COLUMNS)}

def build_energy_tensors(
    runs_dir: Path = RUNS_DIR,
    ampar_values: Optional[Iterable[float]] = None,
    rin_values: Optional[Iterable[float]] = None,
    eta_values: Optional[Iterable[float]] = None,
    temp_values: Optional[Iterable[float]] = None,
    sigma_theta_values: Optional[Iterable[float]] = None,
    idir_values: Optional[Iterable[float]] = None,
    trials: Optional[Iterable[int]] = None,
    active_fraction_values: Optional[Iterable[float]] = None,
    recurrent_width_values: Optional[Iterable[float]] = None,
    strict: bool = False,
    tol: float = 1e-12,
) -> Tuple[Dict[Tuple[float, float, float, float, float, float, float], np.ndarray], Dict[str, int]]:
    """Load per-generation synaptic energy arrays into tensors keyed like `tensors`.

    Returns:
      energy_tensors: dict keyed by
        (ampar, rin, sigma_temp, sigma_eta, sigma_theta, recurrent_width, active_fraction)
        with arrays of shape (generations, trials, idir, 4)
      stats: summary counters for skipped/loaded runs.
    """

    def sorted_unique(values: Iterable[float]) -> List[float]:
        return sorted({v for v in values if v is not None})

    def in_allowed(value, allowed: Optional[Iterable[float]]) -> bool:
        if allowed is None:
            return True
        if value is None:
            return False
        if isinstance(value, (float, int)):
            return any(math.isclose(float(value), float(v), rel_tol=tol, abs_tol=tol) for v in allowed)
        return value in allowed

    def get_active_fraction(spec: dict, net: dict):
        active = spec.get("active_fraction")
        if active is None:
            active = net.get("threshold_active_fraction")
        return active

    def get_recurrent_width(spec: dict, net: dict):
        width = spec.get("cann_width")
        if width is None:
            width = net.get("recurrent_width_fraction")
        if width is None:
            width = spec.get("threshold")
        if width is None:
            width = net.get("threshold_active_fraction")
        return width

    def resolve_energy_path(run_dir: Path, net: dict) -> Optional[Path]:
        explicit = net.get("energy_metrics_path")
        if explicit:
            explicit_path = Path(explicit)
            if explicit_path.exists():
                return explicit_path
        fallback = run_dir / "energy_metrics.npy"
        if fallback.exists():
            return fallback
        return None

    def load_energy_matrix(path: Path) -> np.ndarray:
        arr = np.load(path)
        if arr.ndim != 2:
            raise ValueError(f"Expected 2-D energy array at {path}, got shape {arr.shape}")
        if arr.shape[1] != len(ENERGY_COLUMNS):
            raise ValueError(
                f"Expected {len(ENERGY_COLUMNS)} energy columns at {path}, got shape {arr.shape}"
            )
        return arr

    meta_paths = sorted(runs_dir.glob("*/run_metadata.json"))
    if not meta_paths:
        raise FileNotFoundError(f"No run_metadata.json found under {runs_dir}")

    metas = [(p.parent, json.loads(p.read_text())) for p in meta_paths]
    specs = [m.get("spec", {}) for _, m in metas]
    net_params = [m.get("network_params", {}) for _, m in metas]

    if ampar_values is None:
        ampar_values = sorted_unique(s.get("ampar") for s in specs)
    else:
        ampar_values = list(ampar_values)
    if rin_values is None:
        rin_values = sorted_unique(s.get("rin") for s in specs)
    else:
        rin_values = list(rin_values)
    if eta_values is None:
        eta_values = sorted_unique(s.get("sigma_eta") for s in specs)
    else:
        eta_values = list(eta_values)
    if temp_values is None:
        temp_values = sorted_unique(s.get("sigma_temp") for s in specs)
    else:
        temp_values = list(temp_values)
    if sigma_theta_values is None:
        sigma_theta_values = sorted_unique(s.get("sigma_theta") for s in specs)
    else:
        sigma_theta_values = list(sigma_theta_values)
    if idir_values is None:
        idir_values = sorted_unique(s.get("idir") for s in specs)
    else:
        idir_values = list(idir_values)
    if trials is None:
        trials = sorted_unique(s.get("trial") for s in specs)
    else:
        trials = list(trials)

    detected_active = sorted_unique(get_active_fraction(spec, net) for spec, net in zip(specs, net_params))
    active_fraction_filter = None if active_fraction_values is None else list(active_fraction_values)
    active_fraction_values = detected_active if active_fraction_values is None else active_fraction_filter

    detected_width = sorted_unique(get_recurrent_width(spec, net) for spec, net in zip(specs, net_params))
    recurrent_width_filter = None if recurrent_width_values is None else list(recurrent_width_values)
    recurrent_width_values = detected_width if recurrent_width_values is None else recurrent_width_filter

    stats = {
        "scanned_runs": len(metas),
        "matched_metadata": 0,
        "loaded_runs": 0,
        "missing_energy_file": 0,
        "bad_energy_shape": 0,
        "metadata_mismatch": 0,
        "trimmed_for_mismatched_generations": 0,
    }

    runs_by_key: Dict[Tuple[float, float, float, float, float, float, float], Dict[Tuple[float, int], Path]] = {}

    for run_dir, meta in metas:
        spec = meta.get("spec", {})
        net = meta.get("network_params", {})

        ampar = spec.get("ampar")
        rin = spec.get("rin")
        sigma_temp = spec.get("sigma_temp")
        sigma_eta = spec.get("sigma_eta")
        sigma_theta = spec.get("sigma_theta")
        idir = spec.get("idir")
        trial = spec.get("trial")
        active_fraction = get_active_fraction(spec, net)
        recurrent_width = get_recurrent_width(spec, net)

        if None in (ampar, rin, sigma_temp, sigma_eta, sigma_theta, idir, trial, active_fraction, recurrent_width):
            stats["metadata_mismatch"] += 1
            continue

        if not in_allowed(ampar, ampar_values):
            continue
        if not in_allowed(rin, rin_values):
            continue
        if not in_allowed(sigma_temp, temp_values):
            continue
        if not in_allowed(sigma_eta, eta_values):
            continue
        if not in_allowed(sigma_theta, sigma_theta_values):
            continue
        if not in_allowed(idir, idir_values):
            continue
        if not in_allowed(trial, trials):
            continue
        if recurrent_width_filter is not None and not in_allowed(recurrent_width, recurrent_width_filter):
            continue
        if active_fraction_filter is not None and not in_allowed(active_fraction, active_fraction_filter):
            continue

        key = (ampar, rin, sigma_temp, sigma_eta, sigma_theta, recurrent_width, active_fraction)
        runs_by_key.setdefault(key, {})[(idir, trial)] = run_dir
        stats["matched_metadata"] += 1

    energy_tensors: Dict[Tuple[float, float, float, float, float, float, float], np.ndarray] = {}

    for key, mapping in runs_by_key.items():
        idir_blocks = []
        for idir in idir_values:
            trial_blocks = []
            for trial in trials:
                folder = mapping.get((idir, trial))
                if folder is None:
                    if strict:
                        raise KeyError(f"Missing run for key={key}, idir={idir}, trial={trial}")
                    continue

                run_meta = json.loads((folder / "run_metadata.json").read_text())
                energy_path = resolve_energy_path(folder, run_meta.get("network_params", {}))
                if energy_path is None:
                    stats["missing_energy_file"] += 1
                    if strict:
                        raise FileNotFoundError(
                            f"No energy metrics file found for run {folder}. "
                            "Expected network_params['energy_metrics_path'] or energy_metrics.npy"
                        )
                    continue

                try:
                    trial_blocks.append(load_energy_matrix(energy_path))
                    stats["loaded_runs"] += 1
                except ValueError:
                    stats["bad_energy_shape"] += 1
                    if strict:
                        raise
                    continue

            if not trial_blocks:
                if strict:
                    raise KeyError(f"No energy trials found for key={key}, idir={idir}")
                continue

            trial_lengths = [a.shape[0] for a in trial_blocks]
            min_trial_len = min(trial_lengths)
            if any(length != min_trial_len for length in trial_lengths):
                stats["trimmed_for_mismatched_generations"] += 1
            trial_cube = np.stack([a[:min_trial_len, :] for a in trial_blocks], axis=1)  # G x trials x 4
            idir_blocks.append(trial_cube)

        if not idir_blocks:
            if strict:
                raise KeyError(f"No energy idir blocks found for key={key}")
            continue

        idir_lengths = [a.shape[0] for a in idir_blocks]
        min_idir_len = min(idir_lengths)
        if any(length != min_idir_len for length in idir_lengths):
            stats["trimmed_for_mismatched_generations"] += 1
        energy_tensors[key] = np.stack([a[:min_idir_len, :, :] for a in idir_blocks], axis=2)  # G x trials x idir x 4

    print("[energy] load summary:")
    print(
        "  scanned={scanned_runs}, matched_meta={matched_metadata}, loaded_runs={loaded_runs}, "
        "missing_file={missing_energy_file}, bad_shape={bad_energy_shape}, metadata_mismatch={metadata_mismatch}, "
        "trimmed={trimmed_for_mismatched_generations}, condition_blocks={blocks}".format(
            blocks=len(energy_tensors),
            **stats,
        )
    )

    return energy_tensors, stats

def aggregate_energy_curve(
    energy_tensors: Dict[Tuple[float, float, float, float, float, float, float], np.ndarray],
    *,
    ampar: float,
    rin: float,
    sigma_temp: float,
    sigma_eta: float,
    sigma_theta: float,
    recurrent_width: float,
    active_fraction: float,
    metric: str = "sum_abs_total_drive",
) -> Dict[str, np.ndarray]:
    if metric not in ENERGY_COL_IDX:
        raise KeyError(f"Unknown metric '{metric}'. Valid metrics: {list(ENERGY_COL_IDX)}")

    key = (ampar, rin, sigma_temp, sigma_eta, sigma_theta, recurrent_width, active_fraction)
    if key not in energy_tensors:
        raise KeyError(f"Energy key not found: {key}")

    arr = energy_tensors[key]  # G x trials x idir x 4
    metric_values = arr[:, :, :, ENERGY_COL_IDX[metric]]
    traces = metric_values.reshape(metric_values.shape[0], -1)  # G x (trials*idir)

    mean = traces.mean(axis=1)
    if traces.shape[1] > 1:
        sem = traces.std(axis=1, ddof=1) / np.sqrt(traces.shape[1])
    else:
        sem = np.zeros(traces.shape[0], dtype=float)

    return {
        "generation": np.arange(traces.shape[0], dtype=int),
        "mean": mean,
        "sem": sem,
        "n_runs": traces.shape[1],
    }


In [ ]:
PLOT_TEMP = TEMP_VALUES[0]
PLOT_ETA = ETA_VALUES[0]
PLOT_SIGMA_THETA = SIG_THETA_VALUES[0]
PLOT_RECURRENT_WIDTH = RECURRENT_WIDTH_VALUES[0]

CTR_ACTIVE_FRACTION = ACTIVE_FRACTION_VALUES[0]
FR_ACTIVE_FRACTION = ACTIVE_FRACTION_VALUES[0]#1
CTR_PAIR = (1.0, 1.0)
FR_PAIR = (0.71, 1.22)
FR_ALTERNATE_PAIR = (0.86, 1.0)

PLOT_METRIC = "sum_abs_total_drive"

energy_tensors, energy_stats = build_energy_tensors(
    runs_dir=RUNS_DIR,
    temp_values=[PLOT_TEMP],
    eta_values=[PLOT_ETA],
    sigma_theta_values=[PLOT_SIGMA_THETA],
    recurrent_width_values=[PLOT_RECURRENT_WIDTH],
    strict=False,
)

if not energy_tensors:
    raise RuntimeError("No energy metrics found. Re-run simulations with --save-energy-metrics")

print(f"Loaded {len(energy_tensors)} energy condition blocks.")


In [ ]:
ctr_curve = aggregate_energy_curve(
    energy_tensors,
    ampar=CTR_PAIR[0],
    rin=CTR_PAIR[1],
    sigma_temp=PLOT_TEMP,
    sigma_eta=PLOT_ETA,
    sigma_theta=PLOT_SIGMA_THETA,
    recurrent_width=PLOT_RECURRENT_WIDTH,
    active_fraction=CTR_ACTIVE_FRACTION,
    metric=PLOT_METRIC,
)

fr_curve = aggregate_energy_curve(
    energy_tensors,
    ampar=FR_PAIR[0],
    rin=FR_PAIR[1],
    sigma_temp=PLOT_TEMP,
    sigma_eta=PLOT_ETA,
    sigma_theta=PLOT_SIGMA_THETA,
    recurrent_width=PLOT_RECURRENT_WIDTH,
    active_fraction=FR_ACTIVE_FRACTION,
    metric=PLOT_METRIC,
)


### Shared decision-time fit and trajectory calculations

Retains the original 101-value exponent grid and 20 starting points per fit.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.special import ndtr, ndtri

SEPS = np.array([30, 20, 10, 7.5, 5], dtype=float)

OBSERVED = np.array([
    [80.5, 72.5, 68.5, 63.5, 56.5],  # CTR
    [80.0, 72.5, 69.0, 55.5, 52.0],  # FR
])

N_STARTS = 20
RNG_SEED = 7

BETA_GRID = np.linspace(0.25, 1.8, 101)

time_s = np.linspace(0,2,len(fr_snr_rate))
DT = time_s[1] - time_s[0]
bin_snr = np.asarray([
    ctr_snr_rate,
    fr_snr_rate
], dtype=float)

edges = np.r_[time_s, time_s[-1] + DT]

cumulative_snr = np.column_stack([
    np.zeros(2),
    np.cumsum(bin_snr, axis=1),
])

T_MAX = edges[-1]

snr_scale = (
    np.sin(np.deg2rad(SEPS))
    / np.sin(np.deg2rad(SEPS[0]))
) ** 2

difficulty = (
    np.log(SEPS[0] / SEPS)
    / np.log(SEPS[0] / SEPS[-1])
)

def times_from_parameters(p):
    """
    Four fitted time parameters:

        CTR time at 30 deg
        CTR time at 5 deg
        FR  time at 30 deg
        FR  time at 5 deg

    Intermediate times are linear in log difficulty.
    """
    endpoints = np.asarray(p[:4]).reshape(2, 2)

    return (
        endpoints[:, :1] * (1 - difficulty)
        + endpoints[:, 1:] * difficulty
    )

def information_at_times(times):
    """
    Accumulated neural SNR up to each fitted decision time,
    scaled to each angular separation.
    """
    return np.array([
        snr_scale
        * np.interp(times[g], edges, cumulative_snr[g])

        for g in range(2)
    ])

def accuracy(info, log_scale, beta):
    """
    Psychometric transducer:

        P(correct)
          = Phi[
              0.5 * (I / I_scale)^beta
            ]

    beta = 0.5:
        ideal Gaussian/LDA mapping because I = d'^2.

    beta = 1:
        linear mapping from neural SNR to latent
        behavioural discriminability.
    """
    I_scale = np.exp(log_scale)

    latent_dprime = (
        np.maximum(info, 0) / I_scale
    ) ** beta

    return 100 * ndtr(0.5 * latent_dprime)

def initial_log_scale(info, beta):
    """
    Approximate starting value for I_scale obtained by
    algebraically inverting the psychometric function.
    """
    z = ndtri(OBSERVED / 100)

    required_ratio = (2 * z) ** (1 / beta)

    scale_estimates = info / required_ratio

    valid = (
        np.isfinite(scale_estimates)
        & (scale_estimates > 0)
    )

    if not np.any(valid):
        return 0.0

    return np.log(np.median(scale_estimates[valid]))

T_MIN = 0
TIME_BOUNDS = [(T_MIN, T_MAX)] * 4
SCALE_BOUND = [(np.log(1e-6), np.log(1e6))]

BOUNDS = TIME_BOUNDS + SCALE_BOUND

LOWER, UPPER = np.asarray(BOUNDS).T

CONSTRAINTS = [{
    "type": "ineq",
    "fun": lambda p: p[[1, 3]] - p[[0, 2]],
}]

def fit_fixed_beta(beta, seed=RNG_SEED):

    def predict(p):
        times = times_from_parameters(p)
        info = information_at_times(times)

        return accuracy(
            info,
            log_scale=p[4],
            beta=beta,
        )

    def loss(p):
        prediction = predict(p)

        return np.mean((prediction - OBSERVED) ** 2)

    rng = np.random.default_rng(seed)

    solutions = []

    for start in range(N_STARTS):

        if start == 0:
            endpoints = np.array([
                0.12, 0.65,   # CTR
                0.10, 0.45,   # FR
            ])

            endpoints = np.clip(
                endpoints,
                T_MIN,
                T_MAX,
            )

        else:
            endpoints = np.sort(
                rng.uniform(
                    T_MIN,
                    T_MAX,
                    size=(2, 2),
                ),
                axis=1,
            ).ravel()

        initial_info = information_at_times(
            times_from_parameters(endpoints)
        )

        log_scale0 = initial_log_scale(
            initial_info,
            beta,
        )

        p0 = np.r_[
            endpoints,
            log_scale0,
        ]

        p0 = np.clip(
            p0,
            LOWER,
            UPPER,
        )

        result = minimize(
            loss,
            p0,
            method="SLSQP",
            bounds=BOUNDS,
            constraints=CONSTRAINTS,
            options={
                "maxiter": 2000,
                "ftol": 1e-10,
            },
        )

        feasible = np.all(
            result.x[[1, 3]]
            - result.x[[0, 2]]
            >= -1e-7
        )

        if (
            result.success
            and feasible
            and np.all(np.isfinite(result.x))
            and np.isfinite(result.fun)
        ):
            solutions.append(result)

    if not solutions:
        return None

    best = min(
        solutions,
        key=lambda x: x.fun,
    )

    return {
        "beta": beta,
        "fit": best,
        "mse": best.fun,
        "predicted": predict(best.x),
        "times": times_from_parameters(best.x),
        "info": information_at_times(
            times_from_parameters(best.x)
        ),
        "I_scale": np.exp(best.x[4]),
        "n_converged": len(solutions),
    }

profile = []

for beta in BETA_GRID:

    result = fit_fixed_beta(beta)

    if result is None:
        profile.append(np.nan)
    else:
        profile.append(result["mse"])

profile = np.asarray(profile)

if not np.any(np.isfinite(profile)):
    raise RuntimeError(
        "No beta value produced a converged fit."
    )

best_index = np.nanargmin(profile)
best_beta = BETA_GRID[best_index]

best_result = fit_fixed_beta(best_beta)


In [ ]:
def cumulative_energy_on_snr_grid(curve):
    energy = np.r_[0, np.cumsum(curve["mean"])]
    energy_times = np.arange(len(energy)) * dt
    return np.interp(edges[1:], energy_times, energy)

cumulative_energy_ctr = cumulative_energy_on_snr_grid(ctr_curve)

cumulative_energy_fr = cumulative_energy_on_snr_grid(fr_curve)

cumulative_snr_ctr = cumulative_snr[0, 1:]

cumulative_snr_fr = cumulative_snr[1, 1:]


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import ndtr

INTERPOLATE = True

beta = best_result["beta"]
I_scale = best_result["I_scale"]
decision_times = best_result["times"]

energy_reference = cumulative_energy_ctr[-1]

snr_scale_by_sep = (
    np.sin(np.deg2rad(SEPS))
    / np.sin(np.deg2rad(SEPS[0]))
) ** 2

plt.style.use(STYLE_PATH)

COLORS = {
    "FR": "#a82203",
    "FR alternate": "#f1af3a",
    "CTR": "#208cc0",
}

def perf_func(info):
    latent_dprime = (
        np.maximum(info, 0) / I_scale
    ) ** beta

    return 100 * ndtr(0.5 * latent_dprime)

def sample_times_to_stop(stop, interpolate=True):
    """
    Return sampling times from t=0 up to the decision time.

    interpolate=True:
        Append the exact fitted decision time.

    interpolate=False:
        Stop at the last completed bin before the fitted time.
    """
    if interpolate:
        return np.r_[edges[edges < stop], stop]

    final_index = np.searchsorted(
        edges,
        stop,
        side="right",
    ) - 1

    final_index = np.clip(
        final_index,
        0,
        len(edges) - 1,
    )

    return edges[:final_index + 1]

def make_trajectory(
    group_index,
    sep_index,
    cumulative_snr,
    cumulative_energy,
    interpolate=True,
):
    """Return energy, performance, and actual terminal time."""
    stop = decision_times[group_index, sep_index]

    sample_times = sample_times_to_stop(
        stop,
        interpolate=interpolate,
    )

    snr_with_zero = np.r_[0, cumulative_snr]
    energy_with_zero = np.r_[0, cumulative_energy]

    if len(snr_with_zero) != len(edges):
        raise ValueError(
            "SNR and time edges have incompatible lengths: "
            f"{len(snr_with_zero)} versus {len(edges)}."
        )

    if len(energy_with_zero) != len(edges):
        raise ValueError(
            "Energy and time edges have incompatible lengths: "
            f"{len(energy_with_zero)} versus {len(edges)}."
        )

    information = snr_scale_by_sep[sep_index] * np.interp(
        sample_times,
        edges,
        snr_with_zero,
    )

    energy = 100 * np.interp(
        sample_times,
        edges,
        energy_with_zero,
    ) / energy_reference

    performance = perf_func(information)

    return energy, performance, sample_times[-1]

series = [
    (
        "CTR",
        0,
        cumulative_snr_ctr,
        cumulative_energy_ctr,
        COLORS["CTR"],
    ),
    (
        "FR",
        1,
        cumulative_snr_fr,
        cumulative_energy_fr,
        COLORS["FR"],
    ),
]


### Figure 3c: Accuracy versus normalized energy


In [ ]:
fig_final, ax_final = plt.subplots(
    figsize=(mm_to_inch(50), mm_to_inch(40)),
    layout="constrained",
)
final_energies_array = []
final_performances_array = []
for label, g, cum_snr, cum_energy, color in series:
    final_energy = []
    final_performance = []

    for j in range(len(SEPS)):
        energy, performance, actual_stop = make_trajectory(
            group_index=g,
            sep_index=j,
            cumulative_snr=cum_snr,
            cumulative_energy=cum_energy,
            interpolate=INTERPOLATE,
        )

        final_energy.append(energy[-1])
        final_performance.append(performance[-1])

    final_energy = np.asarray(final_energy)*2
    final_performance = np.asarray(final_performance)

    final_energies_array.append(final_energy)
    final_performances_array.append(final_performance)

    ax_final.plot(
        final_energy,
        final_performance,
        marker="o",
        color=color,
        linestyle="--",
        markersize=3,
        label=label,
        zorder=3,

    )

    for x, y, separation in zip(
        final_energy,
        final_performance,
        SEPS,
    ):
        ax_final.annotate(
            f"{separation:g}°",
            xy=(x, y),
            xytext=(3, 3),
            textcoords="offset points",
            fontsize=6,
            color=color,
        )
ax_final.tick_params(axis='both', which='major', labelsize=8)
ax_final.set_xlabel("% of CTR Energy in 1s", fontsize=8)
ax_final.set_ylabel("Accuracy (%)", fontsize=8)
ax_final.legend(frameon=False)
ax_final.set_ylim(49.5,81)
ax_final.axhline(50, color = 'grey', linestyle = '--', linewidth = 0.8)
clean_axis(ax_final)
plt.show()


### Figure 3c inset: SNR per energy versus orientation separation

Uses the final energy arrays calculated for the main panel above and the original
simulation SNR at each fitted decision time.


In [ ]:
def final_snr_at_decision(group_index, cumulative_snr):
    final_snr = []

    for j in range(len(SEPS)):
        sample_times = sample_times_to_stop(
            decision_times[group_index, j],
            interpolate=INTERPOLATE,
        )

        information = snr_scale_by_sep[j] * np.interp(
            sample_times[-1],
            edges,
            np.r_[0, cumulative_snr],
        )
        final_snr.append(information)

    return np.asarray(final_snr)

final_SNR_ctr = final_snr_at_decision(0, cumulative_snr_ctr)
final_SNR_fr = final_snr_at_decision(1, cumulative_snr_fr)

plt.style.use(STYLE_PATH)

fig, ax = plt.subplots(
    figsize=(mm_to_inch(30), mm_to_inch(30)),
    layout="constrained",
)

for label, final_snr, final_energy in [
    ("CTR", final_SNR_ctr, final_energies_array[0]),
    ("FR", final_SNR_fr, final_energies_array[1]),
]:
    ax.plot(
        SEPS,
        final_snr / final_energy,
        color=COLORS[label],
        linestyle="-",
        linewidth=1,
        marker="o",
        markersize=3,
        markeredgewidth=0.8,
        markeredgecolor=COLORS[label],
        label=label,
        zorder=3,
    )

ax.invert_xaxis()
ax.set_xticks(SEPS, [f"{sep:g}" for sep in SEPS])
ax.tick_params(axis="both", which="major", labelsize=8)
ax.set_xlabel(r"$\Delta\theta$ (°)", fontsize=8)
ax.set_ylabel("SNR / Energy", fontsize=8)
clean_axis(ax)
ax.tick_params(axis='y', which='major', labelrotation=90)
plt.show()
